In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# 1. Qwen2.5-3B-Instruct Generation

This section profiles the baseline generation behavior of Qwen2.5-3B-Instruct
on the Kaggle environment before introducing the ART retrieval module.

### Objectives

- Verify that Qwen2.5-3B-Instruct can be loaded successfully on the available GPUs.
- Verify model device placement and data type.
- Measure GPU memory consumption during generation.
- Measure generation latency and token throughput.
- Establish a baseline configuration for subsequent ART experiments.

No ART-specific modification is applied in this section.

In [1]:
import gc
import torch


def cleanup_gpu_memory():
    """
    Release temporary tensors and CUDA allocator state.

    Keeps:
        - model
        - tokenizer
    """

    # Explicitly remove common temporary objects if they exist.
    temp_vars = [
        "inputs",
        "outputs",
        "input_ids",
        "attention_mask",
        "generated_tokens",
        "response",
    ]

    for name in temp_vars:
        if name in globals():
            del globals()[name]

    # Python garbage collection
    gc.collect()

    if torch.cuda.is_available():
        for device_id in range(torch.cuda.device_count()):
            torch.cuda.synchronize(device_id)

        # Release unused cached blocks on all GPUs
        torch.cuda.empty_cache()

        # Release CUDA IPC resources
        torch.cuda.ipc_collect()

        # Reset profiling statistics on all GPUs
        for device_id in range(torch.cuda.device_count()):
            torch.cuda.reset_peak_memory_stats(device_id)
            torch.cuda.reset_accumulated_memory_stats(device_id)

    gc.collect()

## 1.1 Load Qwen2.5-3B-Instruct

Load the Qwen2.5-3B-Instruct causal language model and tokenizer using
automatic device mapping across the available GPUs.

This test measures the GPU memory required to keep the model loaded in
memory, without performing any generation.

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# Clean GPU memory before loading
cleanup_gpu_memory()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model.eval()

# Synchronize all GPUs before measuring memory
for device_id in range(torch.cuda.device_count()):
    torch.cuda.synchronize(device_id)

# Measure VRAM
gpu_memory = []

for device_id in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(device_id) / 1024**3
    reserved = torch.cuda.memory_reserved(device_id) / 1024**3

    gpu_memory.append(allocated)

    print(
        f"GPU {device_id}: "
        f"allocated = {allocated:.2f} GB | "
        f"reserved = {reserved:.2f} GB"
    )

total_allocated = sum(gpu_memory)

print(f"\nTotal allocated VRAM: {total_allocated:.2f} GB")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

GPU 0: allocated = 2.88 GB | reserved = 2.91 GB
GPU 1: allocated = 2.87 GB | reserved = 2.91 GB

Total allocated VRAM: 5.75 GB


## 1.2 Basic Generation

Test the basic instruction-following generation capability of
Qwen2.5-3B-Instruct.

The input follows the standard chat format:

1. Define a simple user message.
2. Apply the model's chat template.
3. Tokenize the formatted prompt.
4. Generate a response using the loaded causal language model.
5. Decode the generated tokens into text.

In [4]:
# Simple user prompt
messages = [
    {
        "role": "user",
        "content": "What is machine learning?"
    }
]

# Apply Qwen's chat template
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print("Formatted prompt:")
print(text)

Formatted prompt:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant



In [5]:
# Tokenize
inputs = tokenizer(
    text,
    return_tensors="pt",
)

# Move inputs to the device of the input embedding layer
input_device = model.get_input_embeddings().weight.device
inputs = {key: value.to(input_device) for key, value in inputs.items()}

print(f"Input device: {input_device}")
print(f"Input tokens: {inputs['input_ids'].shape[-1]}")

Input device: cuda:0
Input tokens: 34


In [6]:
# Generate
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
    )

# Extract only newly generated tokens
generated_tokens = outputs[0, inputs["input_ids"].shape[-1]:]

# Decode
response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
)

print("Generated response:")
print(response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generated response:
Machine learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enable computer systems to improve their performance on a specific task through experience or data without being explicitly programmed.

In simpler terms, machine learning involves training computers to recognize patterns in data and make decisions or predictions based on those patterns. The process typically involves three main components:

1. **Data**: This is the input used to train the machine learning model. It can be structured (like a database) or unstructured (like text, images, or audio).

2. **Model**: This is the algorithm that learns from the data. There


## 1.3 Chain-of-Thought Generation

Test the model's ability to generate a step-by-step reasoning process using an R-Search-style instruction.

At this stage, no external retrieval or search tool is provided. The test only evaluates whether the instruction-tuned model can produce explicit multi-step reasoning before giving the final answer.

In [10]:
# R-Search-style CoT prompt

text = """You are a helpful assistant that can solve the given question step by step.
For each step, start by explaining your thought process.
When you know the final answer, provide the final answer within <answer> and </answer> tags.

A train travels 120 km in 2 hours. It then travels another 180 km in 3 hours.
What is the average speed of the train over the entire journey?"""

# Apply Qwen chat template
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": text}],
    tokenize=False,
    add_generation_prompt=True,
)

# Tokenize
inputs = tokenizer(
    text,
    return_tensors="pt",
)

# Move inputs to the device of the input embedding layer
input_device = model.get_input_embeddings().weight.device
inputs = {key: value.to(input_device) for key, value in inputs.items()}

print(f"Input device: {input_device}")
print(f"Input tokens: {inputs['input_ids'].shape[-1]}")

# Generate
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
    )

# Extract only newly generated tokens
generated_tokens = outputs[0, inputs["input_ids"].shape[-1]:]

# Decode
response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
)

print("Generated response:")
print(response)

Input device: cuda:0
Input tokens: 116
Generated response:
To find the average speed of the train over the entire journey, we need to follow these steps:

1. Calculate the total distance traveled by the train.
2. Calculate the total time taken for the journey.
3. Use the formula for average speed: \(\text{Average Speed} = \frac{\text{Total Distance}}{\text{Total Time}}\).

Let's go through each step in detail.

### Step 1: Calculate the Total Distance Traveled

The train first travels 120 km and then another 180 km.

\[ \text{Total Distance} = 120 \, \text{km} + 180 \, \text{km} = 300 \, \text{km} \]

### Step 2: Calculate the Total Time Taken

The train takes 2 hours for the first part of the journey and 3 hours for the second part.

\[ \text{Total Time} = 2 \, \text{hours} + 3 \, \text{hours} = 5 \, \text{hours} \]

### Step 3: Calculate the Average Speed

Now, we use the formula for average speed:

\[ \text{Average Speed} = \frac{\text{Total Distance}}{\text{Total Time}} = \frac{300

## 1.4 Multi-Hop Reasoning over Context

Test the model's ability to perform multi-hop reasoning over information provided directly in the context.

The question requires combining information from multiple entries in a diary rather than solving a mathematical problem or relying on external knowledge.

In [15]:
# Multi-hop reasoning over a diary context

system_prompt = """You are a helpful assistant that can solve the given question step by step.
For each step, start by explaining your thought process.
When you know the final answer, provide the final answer within <answer> and </answer> tags."""

user_prompt = """Here is a page from Alex's diary:

Monday:
I met my old friend Daniel at the library. He told me that he had recently started working at a small publishing company called Maple Press.

Tuesday:
I went to the coffee shop near the library. Daniel was there with his colleague Sarah. She was carrying a box of books from Maple Press.

Wednesday:
Sarah invited me to a book discussion at the community center. She said the event was organized by her employer.

Thursday:
I received an email from Maple Press about the book discussion. The email was signed by Daniel's colleague.

Question:
Where does Sarah work, and how do you know that?"""

# Apply Qwen chat template
text = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    tokenize=False,
    add_generation_prompt=True,
)

# Tokenize
inputs = tokenizer(
    text,
    return_tensors="pt",
)

# Move inputs to the device of the input embedding layer
input_device = model.get_input_embeddings().weight.device
inputs = {key: value.to(input_device) for key, value in inputs.items()}

print(f"Input device: {input_device}")
print(f"Input tokens: {inputs['input_ids'].shape[-1]}")

# Generate
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
    )

# Extract only newly generated tokens
generated_tokens = outputs[0, inputs["input_ids"].shape[-1]:]

# Decode
response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
)

print("Generated response:")
print(response)

Input device: cuda:0
Input tokens: 191
Generated response:
To determine where Sarah works, let's analyze the information provided in Alex's diary step-by-step.

1. **Monday**: Alex meets Daniel at the library. Daniel mentions he has started working at a small publishing company called Maple Press.
2. **Tuesday**: Alex goes to the coffee shop near the library where he finds Daniel and his colleague Sarah. Sarah is carrying a box of books from Maple Press.
3. **Wednesday**: Sarah invites Alex to a book discussion at the community center. She says the event is organized by her employer.
4. **Thursday**: Alex receives an email from Maple Press about the book discussion. The email is signed by Daniel's colleague.

From these steps, we can deduce the following:
- On Monday, Daniel mentions he has started working at Maple Press.
- On Tuesday, Sarah is carrying a box of books from Maple Press, indicating she is involved with Maple Press.
- On Wednesday, Sarah organizes a book discussion for he

## 1.5 CoT Generation with Retrieved Context

Test the model's ability to continue step-by-step reasoning after receiving retrieved information.

The prompt is adapted from the R-Search reasoning format:
- The model generates step-by-step reasoning.
- When additional information is needed, the system performs retrieval.
- Retrieved information is returned within `<observation>` and `</observation>` tags.
- The model then continues its reasoning using the retrieved information.
- The final answer is provided within `<answer>` and `</answer>` tags.

At this stage, retrieval is simulated by directly providing a retrieved diary passage in the assistant context.

In [18]:
# R-Search-style reasoning with retrieved context

system_prompt = """You are a helpful assistant that can solve the given question step by step.
For each step, start by explaining your thought process.
If additional information is needed, the system will perform retrieval and return the top search results within <observation> and </observation>. Continue your reasoning using the retrieved information.
When you know the final answer, provide the final answer within <answer> and </answer> tags."""

user_prompt = """Question:
Where does Sarah work, and how do you know that?"""

retrieved_context = """Here is a page from Alex's diary:

Monday:
I met my old friend Daniel at the library. He told me that he had recently started working at a small publishing company called Maple Press.

Tuesday:
I went to the coffee shop near the library. Daniel was there with his colleague Sarah. She was carrying a box of books from Maple Press.

Wednesday:
Sarah invited me to a book discussion at the community center. She said the event was organized by her employer.

Thursday:
I received an email from Maple Press about the book discussion. The email was signed by Daniel's colleague."""

# Apply Qwen chat template up to the assistant generation point
text = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    tokenize=False,
    add_generation_prompt=True,
)

# Inject retrieved observation directly into the assistant generation context
text += "<observation>\n" + retrieved_context + "\n</observation>\n"

print("=" * 80)
print("MODEL INPUT:")
print("=" * 80)
print(text)
print("=" * 80)

# Tokenize
inputs = tokenizer(
    text,
    return_tensors="pt",
)

# Move inputs to the device of the input embedding layer
input_device = model.get_input_embeddings().weight.device
inputs = {key: value.to(input_device) for key, value in inputs.items()}

print(f"Input device: {input_device}")
print(f"Input tokens: {inputs['input_ids'].shape[-1]}")

# Generate
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
    )

# Extract only newly generated tokens
generated_tokens = outputs[0, inputs["input_ids"].shape[-1]:]

# Decode
response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
)

print("Generated response:")
print(response)

MODEL INPUT:
<|im_start|>system
You are a helpful assistant that can solve the given question step by step.
For each step, start by explaining your thought process.
If additional information is needed, the system will perform retrieval and return the top search results within <observation> and </observation>. Continue your reasoning using the retrieved information.
When you know the final answer, provide the final answer within <answer> and </answer> tags.<|im_end|>
<|im_start|>user
Question:
Where does Sarah work, and how do you know that?<|im_end|>
<|im_start|>assistant
<observation>
Here is a page from Alex's diary:

Monday:
I met my old friend Daniel at the library. He told me that he had recently started working at a small publishing company called Maple Press.

Tuesday:
I went to the coffee shop near the library. Daniel was there with his colleague Sarah. She was carrying a box of books from Maple Press.

Wednesday:
Sarah invited me to a book discussion at the community center. S

## 1.6 Multi-Hop Reasoning with Iterative Retrieval

Test the model's ability to perform multi-hop reasoning across multiple retrieval rounds.

Each retrieval round returns multiple passages, simulating a top-3 retrieval setting. The model first reasons over the information from the initial retrieval, then receives another set of retrieved passages and continues its reasoning before producing the final answer.

This simulates the interleaved reasoning and retrieval process used in adaptive retrieval systems.

In [19]:
# Multi-hop reasoning with two retrieval rounds

system_prompt = """You are a helpful assistant that can solve the given question step by step.
For each step, start by explaining your thought process.
If additional information is needed, the system will perform retrieval and return the top search results within <observation> and </observation>. Continue your reasoning using the retrieved information.
When you know the final answer, provide the final answer within <answer> and </answer> tags."""

user_prompt = """Question:
What company did Sarah's colleague Daniel start working for, and what type of company is it?"""

# First retrieval: top-3 results
retrieval_1 = """<observation>

[1] Title: Daniel's New Job
Daniel recently started working at a small publishing company called Maple Press.

[2] Title: Meeting at the Library
Daniel met Alex at the library on Monday and told him about his new job.

[3] Title: Maple Press Book Collection
Maple Press recently announced a new collection of historical novels.

</observation>
"""

# Partial generation between the two retrieval rounds
partial_generation = """Based on the retrieved information, Daniel recently started working at Maple Press.
I now need to determine what type of company Maple Press is.

"""

# Second retrieval: top-3 results
retrieval_2 = """<observation>

[1] Title: About Maple Press
Maple Press is an independent publishing company that specializes in books and literary publications.

[2] Title: Sarah and Daniel at Maple Press
Sarah works with Daniel at Maple Press and often helps organize book-related events.

[3] Title: History of Maple Press
Maple Press was founded to support emerging authors and publish new literary works.

</observation>
"""

# Apply Qwen chat template up to the assistant generation point
text = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    tokenize=False,
    add_generation_prompt=True,
)

# Inject the entire multi-hop trajectory directly into the assistant generation context
text += retrieval_1
text += partial_generation
text += retrieval_2

print("=" * 80)
print("MODEL INPUT:")
print("=" * 80)
print(text)
print("=" * 80)

# Tokenize
inputs = tokenizer(
    text,
    return_tensors="pt",
)

# Move inputs to the device of the input embedding layer
input_device = model.get_input_embeddings().weight.device
inputs = {key: value.to(input_device) for key, value in inputs.items()}

print(f"Input device: {input_device}")
print(f"Input tokens: {inputs['input_ids'].shape[-1]}")

# Generate
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
    )

# Extract only newly generated tokens
generated_tokens = outputs[0, inputs["input_ids"].shape[-1]:]

# Decode
response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
)

print("Generated response:")
print(response)

MODEL INPUT:
<|im_start|>system
You are a helpful assistant that can solve the given question step by step.
For each step, start by explaining your thought process.
If additional information is needed, the system will perform retrieval and return the top search results within <observation> and </observation>. Continue your reasoning using the retrieved information.
When you know the final answer, provide the final answer within <answer> and </answer> tags.<|im_end|>
<|im_start|>user
Question:
What company did Sarah's colleague Daniel start working for, and what type of company is it?<|im_end|>
<|im_start|>assistant
<observation>

[1] Title: Daniel's New Job
Daniel recently started working at a small publishing company called Maple Press.

[2] Title: Meeting at the Library
Daniel met Alex at the library on Monday and told him about his new job.

[3] Title: Maple Press Book Collection
Maple Press recently announced a new collection of historical novels.

</observation>
Based on the retri

# 2. VRAM Profiling

This section profiles GPU memory usage under the training configuration used by ART
before running the full training process.

### Objectives

- Verify the available GPU memory and device configuration.
- Measure GPU memory consumption of the frozen Qwen2.5-3B-Instruct backbone.
- Measure activation and gradient memory during the training process.
- Measure optimizer memory for the trainable ART retrieval-decision module.
- Measure peak GPU memory during forward and backward passes.
- Establish a baseline VRAM configuration for subsequent ART experiments.

No full ART training is performed in this section.

## 2.1. Full Model Training with Gradient Checkpointing

This experiment profiles the memory and time requirements of full-model training
for Qwen2.5-3B-Instruct with gradient checkpointing enabled.

The model is fully unfrozen and trained with an input sequence length of 1024 tokens
using the Hugging Face Trainer.
### Objectives

- Measure peak VRAM usage on both GPUs during full-model training.
- Measure training time for a single training step.
- Evaluate the effect of gradient checkpointing on memory consumption.
- Verify that the full Qwen2.5-3B-Instruct model can perform forward and backward
  passes under the available 2-GPU Kaggle environment.
- Establish a VRAM and training-time baseline for comparison with the
  parameter-efficient ART training configuration.

In [2]:
import time
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
)

# ============================================================================
# 2.1. Full Model Training with Gradient Checkpointing
# ============================================================================

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

SEQ_LEN = 1024
BATCH_SIZE = 1

# ---------------------------------------------------------------------------
# Cleanup previous inference state
# ---------------------------------------------------------------------------
cleanup_gpu_memory()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Full unfreeze
for param in model.parameters():
    param.requires_grad = True

model.train()

# ---------------------------------------------------------------------------
# Dummy dataset: exactly 1024 tokens
# ---------------------------------------------------------------------------
class DummyDataset(Dataset):
    def __init__(self, tokenizer, seq_len):
        self.input_ids = torch.randint(
            low=0,
            high=tokenizer.vocab_size,
            size=(seq_len,),
            dtype=torch.long,
        )

    def __len__(self):
        return 1

    def __getitem__(self, idx):
        input_ids = self.input_ids.clone()

        return {
            "input_ids": input_ids,
            "attention_mask": torch.ones_like(input_ids),
            "labels": input_ids.clone(),
        }


dataset = DummyDataset(
    tokenizer=tokenizer,
    seq_len=SEQ_LEN,
)

# ---------------------------------------------------------------------------
# Trainer configuration
# ---------------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./tmp_vram_profile",
    
    # One training step
    per_device_train_batch_size=BATCH_SIZE,
    max_steps=1,

    # Gradient checkpointing through HF Trainer
    gradient_checkpointing=True,

    # Precision
    bf16=True,

    # Disable unnecessary features
    logging_strategy="no",
    save_strategy="no",
    report_to="none",

    # Keep dataloader simple
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

# ---------------------------------------------------------------------------
# Reset peak-memory statistics immediately before training
# ---------------------------------------------------------------------------
cleanup_gpu_memory()

print("=" * 80)
print("BEFORE TRAINING")
print("=" * 80)

for device_id in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(device_id) / 1024**3
    reserved = torch.cuda.memory_reserved(device_id) / 1024**3

    print(
        f"GPU {device_id}: "
        f"allocated={allocated:.2f} GB, "
        f"reserved={reserved:.2f} GB"
    )

# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
start_time = time.perf_counter()

trainer.train()

# Synchronize all GPUs before measuring elapsed time
for device_id in range(torch.cuda.device_count()):
    torch.cuda.synchronize(device_id)

elapsed_time = time.perf_counter() - start_time

# ---------------------------------------------------------------------------
# Peak VRAM measurement
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("TRAINING RESULTS")
print("=" * 80)

total_peak_allocated = 0.0
total_peak_reserved = 0.0

for device_id in range(torch.cuda.device_count()):
    peak_allocated = torch.cuda.max_memory_allocated(device_id) / 1024**3
    peak_reserved = torch.cuda.max_memory_reserved(device_id) / 1024**3

    total_peak_allocated += peak_allocated
    total_peak_reserved += peak_reserved

    print(
        f"GPU {device_id}: "
        f"peak allocated={peak_allocated:.2f} GB, "
        f"peak reserved={peak_reserved:.2f} GB"
    )

print("-" * 80)
print(f"Total peak allocated: {total_peak_allocated:.2f} GB")
print(f"Total peak reserved:  {total_peak_reserved:.2f} GB")
print(f"Training time:        {elapsed_time:.2f} s")
print("=" * 80)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


... (3 outputs left out)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

BEFORE TRAINING
GPU 0: allocated=2.88 GB, reserved=2.91 GB
GPU 1: allocated=2.87 GB, reserved=2.91 GB


Step,Training Loss



TRAINING RESULTS
GPU 0: peak allocated=11.62 GB, peak reserved=11.84 GB
GPU 1: peak allocated=11.64 GB, peak reserved=11.92 GB
--------------------------------------------------------------------------------
Total peak allocated: 23.26 GB
Total peak reserved:  23.77 GB
Training time:        11.26 s


In [3]:
import gc
import torch


def cleanup_gpu_memory():
    """
    Clean temporary GPU objects while keeping:
        - model
        - tokenizer
    """

    # Remove common temporary objects
    temp_vars = [
        "inputs",
        "outputs",
        "input_ids",
        "attention_mask",
        "generated_tokens",
        "response",
        "text",
        "loss",
        "losses",
        "batch",
        "batch_inputs",
        "batch_outputs",
        "logits",
        "labels",
        "hidden_states",
        "past_key_values",
    ]

    for name in temp_vars:
        if name in globals():
            del globals()[name]

    # Python garbage collection
    gc.collect()

    if torch.cuda.is_available():
        for device_id in range(torch.cuda.device_count()):
            torch.cuda.synchronize(device_id)
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

    gc.collect()

    # Reset peak memory statistics
    if torch.cuda.is_available():
        for device_id in range(torch.cuda.device_count()):
            torch.cuda.reset_peak_memory_stats(device_id)
            torch.cuda.reset_accumulated_memory_stats(device_id)

    print("GPU memory after cleanup:")
    for device_id in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(device_id) / 1024**3
        reserved = torch.cuda.memory_reserved(device_id) / 1024**3

        print(
            f"GPU {device_id}: "
            f"allocated={allocated:.2f} GB | "
            f"reserved={reserved:.2f} GB"
        )

In [4]:
cleanup_gpu_memory()

GPU memory after cleanup:
GPU 0: allocated=8.74 GB | reserved=10.06 GB
GPU 1: allocated=8.75 GB | reserved=10.07 GB


In [5]:
del trainer
gc.collect()

torch.cuda.empty_cache()
torch.cuda.ipc_collect()

for device_id in range(torch.cuda.device_count()):
    torch.cuda.synchronize(device_id)

print("GPU memory:")
for device_id in range(torch.cuda.device_count()):
    print(
        f"GPU {device_id}: "
        f"allocated={torch.cuda.memory_allocated(device_id) / 1024**3:.2f} GB | "
        f"reserved={torch.cuda.memory_reserved(device_id) / 1024**3:.2f} GB"
    )

GPU memory:
GPU 0: allocated=2.89 GB | reserved=2.93 GB
GPU 1: allocated=2.89 GB | reserved=2.93 GB


## 2.2 Training with Frozen Embeddings

This experiment repeats the training configuration from Section 2.1,
while freezing the input embedding layer.

The purpose is to measure the effect of freezing the embedding layer
on GPU memory consumption and training time.

In [6]:
import time
import torch
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments

# ============================================================================
# 2.2. Full Model Training with Frozen Embeddings
# ============================================================================

SEQ_LEN = 1024
BATCH_SIZE = 1

# ---------------------------------------------------------------------------
# Cleanup previous training state
# Keep:
#   - model
#   - tokenizer
# ---------------------------------------------------------------------------
if "trainer" in globals():
    del trainer

cleanup_gpu_memory()

# ---------------------------------------------------------------------------
# Freeze input embedding layer
# ---------------------------------------------------------------------------
embedding_layer = model.get_input_embeddings()

for param in embedding_layer.parameters():
    param.requires_grad = False

# Make sure all other model parameters are trainable
for name, param in model.named_parameters():
    if param is not embedding_layer.weight:
        param.requires_grad = True

model.train()

# ---------------------------------------------------------------------------
# Verify trainable / frozen parameters
# ---------------------------------------------------------------------------
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

print("=" * 80)
print("PARAMETER CONFIGURATION")
print("=" * 80)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {frozen_params:,}")
print("-" * 80)
print(
    f"Embedding parameters: "
    f"{embedding_layer.weight.numel():,}"
)
print("=" * 80)

# ---------------------------------------------------------------------------
# Dummy dataset: exactly 1024 tokens
# ---------------------------------------------------------------------------
class DummyDataset(Dataset):
    def __init__(self, tokenizer, seq_len):
        self.input_ids = torch.randint(
            low=0,
            high=tokenizer.vocab_size,
            size=(seq_len,),
            dtype=torch.long,
        )

    def __len__(self):
        return 1

    def __getitem__(self, idx):
        input_ids = self.input_ids.clone()

        return {
            "input_ids": input_ids,
            "attention_mask": torch.ones_like(input_ids),
            "labels": input_ids.clone(),
        }


dataset = DummyDataset(
    tokenizer=tokenizer,
    seq_len=SEQ_LEN,
)

# ---------------------------------------------------------------------------
# Trainer configuration
# Same configuration as Section 2.1
# ---------------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./tmp_vram_profile_2_2",

    # One training step
    per_device_train_batch_size=BATCH_SIZE,
    max_steps=1,

    # Gradient checkpointing
    gradient_checkpointing=True,

    # Precision
    bf16=True,

    # Disable unnecessary features
    logging_strategy="no",
    save_strategy="no",
    report_to="none",

    # Keep dataloader simple
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

# ---------------------------------------------------------------------------
# Reset peak-memory statistics immediately before training
# ---------------------------------------------------------------------------
cleanup_gpu_memory()

print("=" * 80)
print("BEFORE TRAINING — 2.2")
print("=" * 80)

for device_id in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(device_id) / 1024**3
    reserved = torch.cuda.memory_reserved(device_id) / 1024**3

    print(
        f"GPU {device_id}: "
        f"allocated={allocated:.2f} GB, "
        f"reserved={reserved:.2f} GB"
    )

# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
start_time = time.perf_counter()

trainer.train()

# Synchronize all GPUs before measuring elapsed time
for device_id in range(torch.cuda.device_count()):
    torch.cuda.synchronize(device_id)

elapsed_time = time.perf_counter() - start_time

# ---------------------------------------------------------------------------
# Peak VRAM measurement
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("TRAINING RESULTS — 2.2")
print("=" * 80)

total_peak_allocated = 0.0
total_peak_reserved = 0.0

for device_id in range(torch.cuda.device_count()):
    peak_allocated = (
        torch.cuda.max_memory_allocated(device_id)
        / 1024**3
    )

    peak_reserved = (
        torch.cuda.max_memory_reserved(device_id)
        / 1024**3
    )

    total_peak_allocated += peak_allocated
    total_peak_reserved += peak_reserved

    print(
        f"GPU {device_id}: "
        f"peak allocated={peak_allocated:.2f} GB, "
        f"peak reserved={peak_reserved:.2f} GB"
    )

print("-" * 80)
print(
    f"Total peak allocated: {total_peak_allocated:.2f} GB"
)
print(
    f"Total peak reserved:  {total_peak_reserved:.2f} GB"
)
print(
    f"Training time:        {elapsed_time:.2f} s"
)
print("=" * 80)

GPU memory after cleanup:
GPU 0: allocated=2.89 GB | reserved=2.93 GB
GPU 1: allocated=2.89 GB | reserved=2.93 GB
PARAMETER CONFIGURATION
Total parameters:     3,085,938,688
Trainable parameters: 2,774,773,760
Frozen parameters:    311,164,928
--------------------------------------------------------------------------------
Embedding parameters: 311,164,928
GPU memory after cleanup:
GPU 0: allocated=2.89 GB | reserved=2.93 GB
GPU 1: allocated=2.89 GB | reserved=2.93 GB
BEFORE TRAINING — 2.2
GPU 0: allocated=2.89 GB, reserved=2.93 GB
GPU 1: allocated=2.89 GB, reserved=2.93 GB


Step,Training Loss



TRAINING RESULTS — 2.2
GPU 0: peak allocated=9.88 GB, peak reserved=10.05 GB
GPU 1: peak allocated=11.64 GB, peak reserved=11.92 GB
--------------------------------------------------------------------------------
Total peak allocated: 21.52 GB
Total peak reserved:  21.98 GB
Training time:        11.25 s


In [7]:
del trainer
gc.collect()

torch.cuda.empty_cache()
torch.cuda.ipc_collect()

for device_id in range(torch.cuda.device_count()):
    torch.cuda.synchronize(device_id)

print("GPU memory:")
for device_id in range(torch.cuda.device_count()):
    print(
        f"GPU {device_id}: "
        f"allocated={torch.cuda.memory_allocated(device_id) / 1024**3:.2f} GB | "
        f"reserved={torch.cuda.memory_reserved(device_id) / 1024**3:.2f} GB"
    )

GPU memory:
GPU 0: allocated=2.89 GB | reserved=2.93 GB
GPU 1: allocated=2.89 GB | reserved=2.93 GB


## 2.3. Partial Model Training with Frozen First Half

This experiment repeats the training configuration from Section 2.1,
while freezing the input embedding layer and the first half of the
transformer layers.

For Qwen2.5-3B, which contains 36 transformer layers:

- **Frozen:** input embedding + transformer layers 0–17
- **Trainable:** transformer layers 18–35 + final normalization
- **LM head:** frozen because it shares weights with the input embedding

The purpose is to measure how freezing the first half of the model
affects GPU memory consumption and training time.

In [8]:
import time
import torch
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments

# ============================================================================
# 2.3. Partial Model Training
# Freeze first half of transformer layers + embedding
# Train second half of transformer layers
# ============================================================================

SEQ_LEN = 1024
BATCH_SIZE = 1

# ---------------------------------------------------------------------------
# Cleanup previous training state
# Keep:
#   - model
#   - tokenizer
# ---------------------------------------------------------------------------
if "trainer" in globals():
    del trainer

cleanup_gpu_memory()

# ---------------------------------------------------------------------------
# Freeze / unfreeze configuration
# ---------------------------------------------------------------------------
# Qwen2.5-3B has 36 transformer layers:
#   Layers  0-17 : frozen
#   Layers 18-35 : trainable
# ---------------------------------------------------------------------------

NUM_LAYERS = len(model.model.layers)
FREEZE_UNTIL = NUM_LAYERS // 2

print(f"Total transformer layers: {NUM_LAYERS}")
print(f"Frozen layers:   0 - {FREEZE_UNTIL - 1}")
print(f"Trainable layers: {FREEZE_UNTIL} - {NUM_LAYERS - 1}")

# ---------------------------------------------------------------------------
# Freeze everything first
# ---------------------------------------------------------------------------
for param in model.parameters():
    param.requires_grad = False

# ---------------------------------------------------------------------------
# Unfreeze second half of transformer layers
# ---------------------------------------------------------------------------
for layer_idx in range(FREEZE_UNTIL, NUM_LAYERS):
    for param in model.model.layers[layer_idx].parameters():
        param.requires_grad = True

# ---------------------------------------------------------------------------
# Unfreeze final normalization
# ---------------------------------------------------------------------------
for param in model.model.norm.parameters():
    param.requires_grad = True

# ---------------------------------------------------------------------------
# NOTE:
# Qwen2.5 uses tied input/output embeddings.
#
# Since the input embedding is frozen, the tied lm_head is also kept frozen.
# ---------------------------------------------------------------------------

model.train()

# ---------------------------------------------------------------------------
# Verify parameter configuration
# ---------------------------------------------------------------------------
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

print("=" * 80)
print("PARAMETER CONFIGURATION")
print("=" * 80)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {frozen_params:,}")

print("-" * 80)

# Per-layer verification
for layer_idx in [0, FREEZE_UNTIL - 1, FREEZE_UNTIL, NUM_LAYERS - 1]:
    layer_trainable = any(
        p.requires_grad
        for p in model.model.layers[layer_idx].parameters()
    )

    status = "TRAINABLE" if layer_trainable else "FROZEN"

    print(
        f"Layer {layer_idx:2d}: {status}"
    )

print("-" * 80)

embedding_trainable = any(
    p.requires_grad
    for p in model.get_input_embeddings().parameters()
)

norm_trainable = any(
    p.requires_grad
    for p in model.model.norm.parameters()
)

lm_head_trainable = any(
    p.requires_grad
    for p in model.lm_head.parameters()
)

print(
    f"Input embedding: "
    f"{'TRAINABLE' if embedding_trainable else 'FROZEN'}"
)

print(
    f"Final norm: "
    f"{'TRAINABLE' if norm_trainable else 'FROZEN'}"
)

print(
    f"LM head: "
    f"{'TRAINABLE' if lm_head_trainable else 'FROZEN'}"
)

print("=" * 80)

# ---------------------------------------------------------------------------
# Dummy dataset: exactly 1024 tokens
# ---------------------------------------------------------------------------
class DummyDataset(Dataset):
    def __init__(self, tokenizer, seq_len):
        self.input_ids = torch.randint(
            low=0,
            high=tokenizer.vocab_size,
            size=(seq_len,),
            dtype=torch.long,
        )

    def __len__(self):
        return 1

    def __getitem__(self, idx):
        input_ids = self.input_ids.clone()

        return {
            "input_ids": input_ids,
            "attention_mask": torch.ones_like(input_ids),
            "labels": input_ids.clone(),
        }


dataset = DummyDataset(
    tokenizer=tokenizer,
    seq_len=SEQ_LEN,
)

# ---------------------------------------------------------------------------
# Trainer configuration
# Same configuration as 2.1 / 2.2
# ---------------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./tmp_vram_profile_2_3",

    # One training step
    per_device_train_batch_size=BATCH_SIZE,
    max_steps=1,

    # Gradient checkpointing
    gradient_checkpointing=True,

    # Precision
    bf16=True,

    # Disable unnecessary features
    logging_strategy="no",
    save_strategy="no",
    report_to="none",

    # Keep dataloader simple
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

# ---------------------------------------------------------------------------
# Reset peak-memory statistics immediately before training
# ---------------------------------------------------------------------------
cleanup_gpu_memory()

print("=" * 80)
print("BEFORE TRAINING — 2.3")
print("=" * 80)

for device_id in range(torch.cuda.device_count()):
    allocated = (
        torch.cuda.memory_allocated(device_id)
        / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved(device_id)
        / 1024**3
    )

    print(
        f"GPU {device_id}: "
        f"allocated={allocated:.2f} GB, "
        f"reserved={reserved:.2f} GB"
    )

# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
start_time = time.perf_counter()

trainer.train()

# Synchronize all GPUs before measuring elapsed time
for device_id in range(torch.cuda.device_count()):
    torch.cuda.synchronize(device_id)

elapsed_time = time.perf_counter() - start_time

# ---------------------------------------------------------------------------
# Peak VRAM measurement
# ---------------------------------------------------------------------------
print("\n" + "=" * 80)
print("TRAINING RESULTS — 2.3")
print("=" * 80)

total_peak_allocated = 0.0
total_peak_reserved = 0.0

for device_id in range(torch.cuda.device_count()):
    peak_allocated = (
        torch.cuda.max_memory_allocated(device_id)
        / 1024**3
    )

    peak_reserved = (
        torch.cuda.max_memory_reserved(device_id)
        / 1024**3
    )

    total_peak_allocated += peak_allocated
    total_peak_reserved += peak_reserved

    print(
        f"GPU {device_id}: "
        f"peak allocated={peak_allocated:.2f} GB, "
        f"peak reserved={peak_reserved:.2f} GB"
    )

print("-" * 80)

print(
    f"Total peak allocated: "
    f"{total_peak_allocated:.2f} GB"
)

print(
    f"Total peak reserved:  "
    f"{total_peak_reserved:.2f} GB"
)

print(
    f"Training time:        "
    f"{elapsed_time:.2f} s"
)

print("=" * 80)

GPU memory after cleanup:
GPU 0: allocated=2.89 GB | reserved=2.93 GB
GPU 1: allocated=2.89 GB | reserved=2.93 GB
Total transformer layers: 36
Frozen layers:   0 - 17
Trainable layers: 18 - 35
PARAMETER CONFIGURATION
Total parameters:     3,085,938,688
Trainable parameters: 1,387,387,904
Frozen parameters:    1,698,550,784
--------------------------------------------------------------------------------
Layer  0: FROZEN
Layer 17: FROZEN
Layer 18: TRAINABLE
Layer 35: TRAINABLE
--------------------------------------------------------------------------------
Input embedding: FROZEN
Final norm: TRAINABLE
LM head: FROZEN
GPU memory after cleanup:
GPU 0: allocated=2.89 GB | reserved=2.93 GB
GPU 1: allocated=2.89 GB | reserved=2.93 GB
BEFORE TRAINING — 2.3
GPU 0: allocated=2.89 GB, reserved=2.93 GB
GPU 1: allocated=2.89 GB, reserved=2.93 GB


Step,Training Loss



TRAINING RESULTS — 2.3
GPU 0: peak allocated=4.70 GB, peak reserved=5.21 GB
GPU 1: peak allocated=10.76 GB, peak reserved=11.02 GB
--------------------------------------------------------------------------------
Total peak allocated: 15.47 GB
Total peak reserved:  16.23 GB
Training time:        10.46 s


In [9]:
del trainer
gc.collect()

torch.cuda.empty_cache()
torch.cuda.ipc_collect()

for device_id in range(torch.cuda.device_count()):
    torch.cuda.synchronize(device_id)

print("GPU memory:")
for device_id in range(torch.cuda.device_count()):
    print(
        f"GPU {device_id}: "
        f"allocated={torch.cuda.memory_allocated(device_id) / 1024**3:.2f} GB | "
        f"reserved={torch.cuda.memory_reserved(device_id) / 1024**3:.2f} GB"
    )

GPU memory:
GPU 0: allocated=2.89 GB | reserved=2.93 GB
GPU 1: allocated=2.89 GB | reserved=2.93 GB


## 2.4. Partial Model Training with 3,000-Token Sequences

This experiment repeats the training configuration from Section 2.3,
while increasing the sequence length from 1,024 to 3,000 tokens.

The model configuration remains unchanged:

- **Frozen:** input embedding + transformer layers 0–17
- **Trainable:** transformer layers 18–35 + final normalization
- **LM head:** frozen
- **Precision:** BF16
- **Batch size:** 1
- **Gradient checkpointing:** enabled

The purpose is to measure the effect of a longer sequence length
on GPU memory consumption and training time.

In [10]:
import time
import torch
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments

# ============================================================================
# 2.4. Partial Model Training with 3,000-Token Sequences
# ============================================================================

SEQ_LEN = 3000
BATCH_SIZE = 1

# ---------------------------------------------------------------------------
# Cleanup previous training state
# Keep:
#   - model
#   - tokenizer
# ---------------------------------------------------------------------------
if "trainer" in globals():
    del trainer

cleanup_gpu_memory()

# ---------------------------------------------------------------------------
# Freeze / unfreeze configuration
# Same as Section 2.3
# ---------------------------------------------------------------------------

NUM_LAYERS = len(model.model.layers)
FREEZE_UNTIL = NUM_LAYERS // 2

# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze second half of transformer layers
for layer_idx in range(FREEZE_UNTIL, NUM_LAYERS):
    for param in model.model.layers[layer_idx].parameters():
        param.requires_grad = True

# Unfreeze final normalization
for param in model.model.norm.parameters():
    param.requires_grad = True

model.train()

# ---------------------------------------------------------------------------
# Verify parameter configuration
# ---------------------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

print("=" * 80)
print("PARAMETER CONFIGURATION")
print("=" * 80)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {frozen_params:,}")

print("-" * 80)

print(f"Sequence length:      {SEQ_LEN}")
print(f"Batch size:           {BATCH_SIZE}")
print(f"Frozen layers:        0 - {FREEZE_UNTIL - 1}")
print(f"Trainable layers:     {FREEZE_UNTIL} - {NUM_LAYERS - 1}")

print("=" * 80)

# ---------------------------------------------------------------------------
# Dummy dataset: exactly 3,000 tokens
# ---------------------------------------------------------------------------

class DummyDataset(Dataset):
    def __init__(self, tokenizer, seq_len):
        self.input_ids = torch.randint(
            low=0,
            high=tokenizer.vocab_size,
            size=(seq_len,),
            dtype=torch.long,
        )

    def __len__(self):
        return 1

    def __getitem__(self, idx):
        input_ids = self.input_ids.clone()

        return {
            "input_ids": input_ids,
            "attention_mask": torch.ones_like(input_ids),
            "labels": input_ids.clone(),
        }


dataset = DummyDataset(
    tokenizer=tokenizer,
    seq_len=SEQ_LEN,
)

# ---------------------------------------------------------------------------
# Trainer configuration
# Same as Section 2.3
# ---------------------------------------------------------------------------

training_args = TrainingArguments(
    output_dir="./tmp_vram_profile_2_4",

    # One training step
    per_device_train_batch_size=BATCH_SIZE,
    max_steps=1,

    # Gradient checkpointing
    gradient_checkpointing=True,

    # Precision
    bf16=True,

    # Disable unnecessary features
    logging_strategy="no",
    save_strategy="no",
    report_to="none",

    # Keep dataloader simple
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

# ---------------------------------------------------------------------------
# Reset peak-memory statistics immediately before training
# ---------------------------------------------------------------------------

cleanup_gpu_memory()

print("=" * 80)
print("BEFORE TRAINING — 2.4")
print("=" * 80)

for device_id in range(torch.cuda.device_count()):
    allocated = (
        torch.cuda.memory_allocated(device_id)
        / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved(device_id)
        / 1024**3
    )

    print(
        f"GPU {device_id}: "
        f"allocated={allocated:.2f} GB, "
        f"reserved={reserved:.2f} GB"
    )

# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------

start_time = time.perf_counter()

trainer.train()

# Synchronize all GPUs before measuring elapsed time
for device_id in range(torch.cuda.device_count()):
    torch.cuda.synchronize(device_id)

elapsed_time = time.perf_counter() - start_time

# ---------------------------------------------------------------------------
# Peak VRAM measurement
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAINING RESULTS — 2.4")
print("=" * 80)

total_peak_allocated = 0.0
total_peak_reserved = 0.0

for device_id in range(torch.cuda.device_count()):
    peak_allocated = (
        torch.cuda.max_memory_allocated(device_id)
        / 1024**3
    )

    peak_reserved = (
        torch.cuda.max_memory_reserved(device_id)
        / 1024**3
    )

    total_peak_allocated += peak_allocated
    total_peak_reserved += peak_reserved

    print(
        f"GPU {device_id}: "
        f"peak allocated={peak_allocated:.2f} GB, "
        f"peak reserved={peak_reserved:.2f} GB"
    )

print("-" * 80)

print(
    f"Total peak allocated: "
    f"{total_peak_allocated:.2f} GB"
)

print(
    f"Total peak reserved:  "
    f"{total_peak_reserved:.2f} GB"
)

print(
    f"Training time:        "
    f"{elapsed_time:.2f} s"
)

print("=" * 80)

GPU memory after cleanup:
GPU 0: allocated=2.89 GB | reserved=2.93 GB
GPU 1: allocated=2.89 GB | reserved=2.93 GB
PARAMETER CONFIGURATION
Total parameters:     3,085,938,688
Trainable parameters: 1,387,387,904
Frozen parameters:    1,698,550,784
--------------------------------------------------------------------------------
Sequence length:      3000
Batch size:           1
Frozen layers:        0 - 17
Trainable layers:     18 - 35
GPU memory after cleanup:
GPU 0: allocated=2.89 GB | reserved=2.93 GB
GPU 1: allocated=2.89 GB | reserved=2.93 GB
BEFORE TRAINING — 2.4
GPU 0: allocated=2.89 GB, reserved=2.93 GB
GPU 1: allocated=2.89 GB, reserved=2.93 GB


Step,Training Loss



TRAINING RESULTS — 2.4
GPU 0: peak allocated=8.19 GB, peak reserved=10.56 GB
GPU 1: peak allocated=10.70 GB, peak reserved=10.88 GB
--------------------------------------------------------------------------------
Total peak allocated: 18.89 GB
Total peak reserved:  21.44 GB
Training time:        33.31 s


In [11]:
del trainer
gc.collect()

torch.cuda.empty_cache()
torch.cuda.ipc_collect()

for device_id in range(torch.cuda.device_count()):
    torch.cuda.synchronize(device_id)

print("GPU memory:")
for device_id in range(torch.cuda.device_count()):
    print(
        f"GPU {device_id}: "
        f"allocated={torch.cuda.memory_allocated(device_id) / 1024**3:.2f} GB | "
        f"reserved={torch.cuda.memory_reserved(device_id) / 1024**3:.2f} GB"
    )

GPU memory:
GPU 0: allocated=2.89 GB | reserved=2.93 GB
GPU 1: allocated=2.89 GB | reserved=2.93 GB


## 2.5. Partial Model Training with 60–100% Depth

This experiment evaluates partial model training by keeping only the
deepest portion of the transformer trainable.

For Qwen2.5-3B-Instruct, the model contains 36 transformer layers.
Taking approximately 60% of the model depth gives a cutoff around
layer 22.

The model configuration is therefore:

- **Frozen:** input embedding + transformer layers 0–21
- **Trainable:** transformer layers 22–35
- **Final normalization:** trainable
- **LM head:** frozen
- **Precision:** BF16
- **Batch size:** 1
- **Sequence length:** 3,000 tokens
- **Gradient checkpointing:** enabled

The purpose of this experiment is to measure whether training only the
deepest ~40% of the transformer can substantially reduce GPU memory
consumption while maintaining a feasible training configuration.

This setup is motivated by prior layer-wise studies suggesting that
more complex and abstract representations tend to become increasingly
accessible in deeper transformer layers. The 60% cutoff is used only as
a practical profiling point rather than as a fixed boundary for
high-level knowledge.

In [12]:
import time
import torch
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments

# ============================================================================
# 2.5. Partial Model Training with 60–100% Depth
# ============================================================================

SEQ_LEN = 3000
BATCH_SIZE = 1

# ---------------------------------------------------------------------------
# Cleanup previous training state
# Keep:
#   - model
#   - tokenizer
# ---------------------------------------------------------------------------

if "trainer" in globals():
    del trainer

cleanup_gpu_memory()

# ---------------------------------------------------------------------------
# Freeze / unfreeze configuration
#
# Qwen2.5-3B-Instruct:
#   Total transformer layers = 36
#
# 60% depth ≈ 0.60 × 36 = 21.6
#
# Therefore:
#   Frozen:     layers 0–21
#   Trainable:  layers 22–35
# ---------------------------------------------------------------------------

NUM_LAYERS = len(model.model.layers)

FREEZE_UNTIL = int(NUM_LAYERS * 0.60)

# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the deepest ~40% of transformer layers
for layer_idx in range(FREEZE_UNTIL, NUM_LAYERS):
    for param in model.model.layers[layer_idx].parameters():
        param.requires_grad = True

# Unfreeze final normalization
for param in model.model.norm.parameters():
    param.requires_grad = True

model.train()

# ---------------------------------------------------------------------------
# Verify parameter configuration
# ---------------------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

print("=" * 80)
print("PARAMETER CONFIGURATION")
print("=" * 80)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {frozen_params:,}")

print("-" * 80)

print(f"Sequence length:      {SEQ_LEN}")
print(f"Batch size:           {BATCH_SIZE}")

print(
    f"Frozen layers:        "
    f"0 - {FREEZE_UNTIL - 1}"
)

print(
    f"Trainable layers:     "
    f"{FREEZE_UNTIL} - {NUM_LAYERS - 1}"
)

print(
    f"Trainable depth:      "
    f"{FREEZE_UNTIL / NUM_LAYERS:.1%} - 100%"
)

print("=" * 80)

# ---------------------------------------------------------------------------
# Dummy dataset: exactly 3,000 tokens
# ---------------------------------------------------------------------------

class DummyDataset(Dataset):
    def __init__(self, tokenizer, seq_len):
        self.input_ids = torch.randint(
            low=0,
            high=tokenizer.vocab_size,
            size=(seq_len,),
            dtype=torch.long,
        )

    def __len__(self):
        return 1

    def __getitem__(self, idx):
        input_ids = self.input_ids.clone()

        return {
            "input_ids": input_ids,
            "attention_mask": torch.ones_like(input_ids),
            "labels": input_ids.clone(),
        }


dataset = DummyDataset(
    tokenizer=tokenizer,
    seq_len=SEQ_LEN,
)

# ---------------------------------------------------------------------------
# Trainer configuration
# Same as Section 2.4
# ---------------------------------------------------------------------------

training_args = TrainingArguments(
    output_dir="./tmp_vram_profile_2_5",

    # One training step
    per_device_train_batch_size=BATCH_SIZE,
    max_steps=1,

    # Gradient checkpointing
    gradient_checkpointing=True,

    # Precision
    bf16=True,

    # Disable unnecessary features
    logging_strategy="no",
    save_strategy="no",
    report_to="none",

    # Keep dataloader simple
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

# ---------------------------------------------------------------------------
# Reset peak-memory statistics immediately before training
# ---------------------------------------------------------------------------

cleanup_gpu_memory()

print("=" * 80)
print("BEFORE TRAINING — 2.5")
print("=" * 80)

for device_id in range(torch.cuda.device_count()):
    allocated = (
        torch.cuda.memory_allocated(device_id)
        / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved(device_id)
        / 1024**3
    )

    print(
        f"GPU {device_id}: "
        f"allocated={allocated:.2f} GB, "
        f"reserved={reserved:.2f} GB"
    )

# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------

start_time = time.perf_counter()

trainer.train()

# Synchronize all GPUs before measuring elapsed time
for device_id in range(torch.cuda.device_count()):
    torch.cuda.synchronize(device_id)

elapsed_time = time.perf_counter() - start_time

# ---------------------------------------------------------------------------
# Peak VRAM measurement
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAINING RESULTS — 2.5")
print("=" * 80)

total_peak_allocated = 0.0
total_peak_reserved = 0.0

for device_id in range(torch.cuda.device_count()):
    peak_allocated = (
        torch.cuda.max_memory_allocated(device_id)
        / 1024**3
    )

    peak_reserved = (
        torch.cuda.max_memory_reserved(device_id)
        / 1024**3
    )

    total_peak_allocated += peak_allocated
    total_peak_reserved += peak_reserved

    print(
        f"GPU {device_id}: "
        f"peak allocated={peak_allocated:.2f} GB, "
        f"peak reserved={peak_reserved:.2f} GB"
    )

print("-" * 80)

print(
    f"Total peak allocated: "
    f"{total_peak_allocated:.2f} GB"
)

print(
    f"Total peak reserved:  "
    f"{total_peak_reserved:.2f} GB"
)

print(
    f"Training time:        "
    f"{elapsed_time:.2f} s"
)

print("=" * 80)

GPU memory after cleanup:
GPU 0: allocated=2.89 GB | reserved=2.93 GB
GPU 1: allocated=2.89 GB | reserved=2.93 GB
PARAMETER CONFIGURATION
Total parameters:     3,085,938,688
Trainable parameters: 1,156,156,928
Frozen parameters:    1,929,781,760
--------------------------------------------------------------------------------
Sequence length:      3000
Batch size:           1
Frozen layers:        0 - 20
Trainable layers:     21 - 35
Trainable depth:      58.3% - 100%
GPU memory after cleanup:
GPU 0: allocated=2.89 GB | reserved=2.93 GB
GPU 1: allocated=2.89 GB | reserved=2.93 GB
BEFORE TRAINING — 2.5
GPU 0: allocated=2.89 GB, reserved=2.93 GB
GPU 1: allocated=2.89 GB, reserved=2.93 GB


Step,Training Loss



TRAINING RESULTS — 2.5
GPU 0: peak allocated=8.19 GB, peak reserved=10.56 GB
GPU 1: peak allocated=9.39 GB, peak reserved=9.54 GB
--------------------------------------------------------------------------------
Total peak allocated: 17.58 GB
Total peak reserved:  20.10 GB
Training time:        32.70 s


In [13]:
del trainer
gc.collect()

torch.cuda.empty_cache()
torch.cuda.ipc_collect()

for device_id in range(torch.cuda.device_count()):
    torch.cuda.synchronize(device_id)

print("GPU memory:")
for device_id in range(torch.cuda.device_count()):
    print(
        f"GPU {device_id}: "
        f"allocated={torch.cuda.memory_allocated(device_id) / 1024**3:.2f} GB | "
        f"reserved={torch.cuda.memory_reserved(device_id) / 1024**3:.2f} GB"
    )

GPU memory:
GPU 0: allocated=2.89 GB | reserved=2.93 GB
GPU 1: allocated=2.89 GB | reserved=2.93 GB


## 2.6. Partial Model Training with 60–100% Depth + Input Embedding

This experiment extends Section 2.5 by additionally unfreezing the input
embedding layer while keeping the deepest portion of the transformer
trainable.

For Qwen2.5-3B-Instruct, the model contains 36 transformer layers. The
configuration is therefore:

- **Frozen:** transformer layers 0–20
- **Trainable:** transformer layers 21–35
- **Input embedding:** trainable
- **Final normalization:** trainable
- **LM head:** trainable through tied input embedding weights
- **Precision:** BF16
- **Batch size:** 1
- **Sequence length:** 3,000 tokens
- **Gradient checkpointing:** enabled

The purpose of this experiment is to measure the additional GPU memory
and training-time cost introduced by unfreezing the input embedding,
while keeping the transformer training region unchanged from Section
2.5.

This experiment also allows us to determine whether the input embedding
layer introduces a substantial memory overhead compared with training
only the deeper transformer layers.

In [14]:
import time
import torch
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments

# ============================================================================
# 2.6. Partial Model Training with 60–100% Depth + Input Embedding
# ============================================================================

SEQ_LEN = 3000
BATCH_SIZE = 1

# ---------------------------------------------------------------------------
# Cleanup previous training state
# Keep:
#   - model
#   - tokenizer
# ---------------------------------------------------------------------------

if "trainer" in globals():
    del trainer

cleanup_gpu_memory()

# ---------------------------------------------------------------------------
# Freeze / unfreeze configuration
#
# Qwen2.5-3B-Instruct:
#   Total transformer layers = 36
#
# Trainable:
#   - Input embedding
#   - Transformer layers 21–35
#   - Final normalization
#
# Frozen:
#   - Transformer layers 0–20
#
# Note:
# Qwen uses tied input embedding / LM head weights.
# Therefore, unfreezing the input embedding also makes the tied LM head
# parameters trainable.
# ---------------------------------------------------------------------------

NUM_LAYERS = len(model.model.layers)

FREEZE_UNTIL = 21

# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# ---------------------------------------------------------------------------
# Unfreeze input embedding
# ---------------------------------------------------------------------------

embedding_layer = model.get_input_embeddings()

for param in embedding_layer.parameters():
    param.requires_grad = True

# ---------------------------------------------------------------------------
# Unfreeze deepest transformer layers
# ---------------------------------------------------------------------------

for layer_idx in range(FREEZE_UNTIL, NUM_LAYERS):
    for param in model.model.layers[layer_idx].parameters():
        param.requires_grad = True

# ---------------------------------------------------------------------------
# Unfreeze final normalization
# ---------------------------------------------------------------------------

for param in model.model.norm.parameters():
    param.requires_grad = True

model.train()

# ---------------------------------------------------------------------------
# Verify parameter configuration
# ---------------------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

print("=" * 80)
print("PARAMETER CONFIGURATION")
print("=" * 80)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {frozen_params:,}")

print("-" * 80)

print(f"Sequence length:      {SEQ_LEN}")
print(f"Batch size:           {BATCH_SIZE}")

print(
    f"Frozen layers:        "
    f"0 - {FREEZE_UNTIL - 1}"
)

print(
    f"Trainable layers:     "
    f"{FREEZE_UNTIL} - {NUM_LAYERS - 1}"
)

print(
    f"Trainable depth:      "
    f"{FREEZE_UNTIL / NUM_LAYERS:.1%} - 100%"
)

print("Input embedding:      TRAINABLE")

# Check LM head
lm_head_trainable = any(
    p.requires_grad
    for p in model.lm_head.parameters()
)

print(
    f"LM head:              "
    f"{'TRAINABLE' if lm_head_trainable else 'FROZEN'}"
)

print("=" * 80)

# ---------------------------------------------------------------------------
# Dummy dataset: exactly 3,000 tokens
# ---------------------------------------------------------------------------

class DummyDataset(Dataset):
    def __init__(self, tokenizer, seq_len):
        self.input_ids = torch.randint(
            low=0,
            high=tokenizer.vocab_size,
            size=(seq_len,),
            dtype=torch.long,
        )

    def __len__(self):
        return 1

    def __getitem__(self, idx):
        input_ids = self.input_ids.clone()

        return {
            "input_ids": input_ids,
            "attention_mask": torch.ones_like(input_ids),
            "labels": input_ids.clone(),
        }


dataset = DummyDataset(
    tokenizer=tokenizer,
    seq_len=SEQ_LEN,
)

# ---------------------------------------------------------------------------
# Trainer configuration
# Same as Section 2.5
# ---------------------------------------------------------------------------

training_args = TrainingArguments(
    output_dir="./tmp_vram_profile_2_6",

    # One training step
    per_device_train_batch_size=BATCH_SIZE,
    max_steps=1,

    # Gradient checkpointing
    gradient_checkpointing=True,

    # Precision
    bf16=True,

    # Disable unnecessary features
    logging_strategy="no",
    save_strategy="no",
    report_to="none",

    # Keep dataloader simple
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

# ---------------------------------------------------------------------------
# Reset peak-memory statistics immediately before training
# ---------------------------------------------------------------------------

cleanup_gpu_memory()

print("=" * 80)
print("BEFORE TRAINING — 2.6")
print("=" * 80)

for device_id in range(torch.cuda.device_count()):
    allocated = (
        torch.cuda.memory_allocated(device_id)
        / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved(device_id)
        / 1024**3
    )

    print(
        f"GPU {device_id}: "
        f"allocated={allocated:.2f} GB, "
        f"reserved={reserved:.2f} GB"
    )

# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------

start_time = time.perf_counter()

trainer.train()

# Synchronize all GPUs before measuring elapsed time
for device_id in range(torch.cuda.device_count()):
    torch.cuda.synchronize(device_id)

elapsed_time = time.perf_counter() - start_time

# ---------------------------------------------------------------------------
# Peak VRAM measurement
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("TRAINING RESULTS — 2.6")
print("=" * 80)

total_peak_allocated = 0.0
total_peak_reserved = 0.0

for device_id in range(torch.cuda.device_count()):
    peak_allocated = (
        torch.cuda.max_memory_allocated(device_id)
        / 1024**3
    )

    peak_reserved = (
        torch.cuda.max_memory_reserved(device_id)
        / 1024**3
    )

    total_peak_allocated += peak_allocated
    total_peak_reserved += peak_reserved

    print(
        f"GPU {device_id}: "
        f"peak allocated={peak_allocated:.2f} GB, "
        f"peak reserved={peak_reserved:.2f} GB"
    )

print("-" * 80)

print(
    f"Total peak allocated: "
    f"{total_peak_allocated:.2f} GB"
)

print(
    f"Total peak reserved:  "
    f"{total_peak_reserved:.2f} GB"
)

print(
    f"Training time:        "
    f"{elapsed_time:.2f} s"
)

print("=" * 80)

GPU memory after cleanup:
GPU 0: allocated=2.89 GB | reserved=2.93 GB
GPU 1: allocated=2.89 GB | reserved=2.93 GB
PARAMETER CONFIGURATION
Total parameters:     3,085,938,688
Trainable parameters: 1,467,321,856
Frozen parameters:    1,618,616,832
--------------------------------------------------------------------------------
Sequence length:      3000
Batch size:           1
Frozen layers:        0 - 20
Trainable layers:     21 - 35
Trainable depth:      58.3% - 100%
Input embedding:      TRAINABLE
LM head:              TRAINABLE
GPU memory after cleanup:
GPU 0: allocated=2.89 GB | reserved=2.93 GB
GPU 1: allocated=2.89 GB | reserved=2.93 GB
BEFORE TRAINING — 2.6
GPU 0: allocated=2.89 GB, reserved=2.93 GB
GPU 1: allocated=2.89 GB, reserved=2.93 GB


Step,Training Loss



TRAINING RESULTS — 2.6
GPU 0: peak allocated=8.20 GB, peak reserved=10.56 GB
GPU 1: peak allocated=9.39 GB, peak reserved=9.54 GB
--------------------------------------------------------------------------------
Total peak allocated: 17.59 GB
Total peak reserved:  20.10 GB
Training time:        33.54 s


# 3. Qwen2.5-3B-Instruct Generation Structure

This section evaluates the generation capability and generation structure of
Qwen2.5-3B-Instruct in the Kaggle environment before introducing the ART
retrieval decision mechanism.

### Objectives

- Verify that Qwen2.5-3B-Instruct can generate coherent responses.
- Verify the correctness of the chat template and generation prompt.
- Measure GPU memory consumption during generation.
- Measure generation latency and token throughput.
- Verify that the model can continue generation after retrieved-context
  injection.
- Establish a baseline generation structure for subsequent ART experiments.

No ART-specific modification is applied in this section.

## 3.1 Load Qwen2.5-3B-Instruct

Load the Qwen2.5-3B-Instruct causal language model and tokenizer using
automatic device mapping across the available GPUs.

This test restores the pretrained model and tokenizer after reconnecting to
the Kaggle environment. No model parameters are modified, and no generation
is performed in this subsection.

In [2]:
import gc
import torch


def cleanup_gpu_memory():
    """
    Release temporary tensors and CUDA allocator state.

    Keeps:
        - model
        - tokenizer
    """

    # Explicitly remove common temporary objects if they exist.
    temp_vars = [
        "inputs",
        "outputs",
        "input_ids",
        "attention_mask",
        "generated_tokens",
        "response",
    ]

    for name in temp_vars:
        if name in globals():
            del globals()[name]

    # Python garbage collection
    gc.collect()

    if torch.cuda.is_available():
        for device_id in range(torch.cuda.device_count()):
            torch.cuda.synchronize(device_id)

        # Release unused cached blocks on all GPUs
        torch.cuda.empty_cache()

        # Release CUDA IPC resources
        torch.cuda.ipc_collect()

        # Reset profiling statistics on all GPUs
        for device_id in range(torch.cuda.device_count()):
            torch.cuda.reset_peak_memory_stats(device_id)
            torch.cuda.reset_accumulated_memory_stats(device_id)

    gc.collect()

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# Clean GPU memory before loading
cleanup_gpu_memory()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model.eval()

# Synchronize all GPUs before measuring memory
for device_id in range(torch.cuda.device_count()):
    torch.cuda.synchronize(device_id)

# Measure VRAM
gpu_memory = []

for device_id in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(device_id) / 1024**3
    reserved = torch.cuda.memory_reserved(device_id) / 1024**3

    gpu_memory.append(allocated)

    print(
        f"GPU {device_id}: "
        f"allocated = {allocated:.2f} GB | "
        f"reserved = {reserved:.2f} GB"
    )

total_allocated = sum(gpu_memory)

print(f"\nTotal allocated VRAM: {total_allocated:.2f} GB")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

GPU 0: allocated = 2.88 GB | reserved = 2.91 GB
GPU 1: allocated = 2.87 GB | reserved = 2.91 GB

Total allocated VRAM: 5.75 GB


In [4]:
# R-Search-style reasoning with retrieved context
# 2 useful documents + 1 noisy document

system_prompt = """You are a helpful assistant that can solve the given question step by step.
For each step, start by explaining your thought process.
If additional information is needed, the system will perform retrieval and return the top search results within <observation> and </observation>. Continue your reasoning using the retrieved information.
When you know the final answer, provide the final answer within <answer> and </answer> tags."""

user_prompt = """Question:
Where does Sarah work, and how do you know that?"""

retrieved_context = """[1] Title: Daniel's New Job

Daniel recently started working at a small publishing company called Maple Press.
He told his friend Alex about his new job at the company.


[2] Title: Sarah and Daniel at Maple Press

Sarah works with Daniel at Maple Press.
She often helps organize book-related events for the company and works with
Daniel on book-related activities.


[3] Title: Weekend Weather Report

The city experienced heavy rainfall over the weekend.
Temperatures remained relatively low, and several outdoor events were
cancelled because of the weather."""

# Apply Qwen chat template up to the assistant generation point
text = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    tokenize=False,
    add_generation_prompt=True,
)

# Inject retrieved observation directly into the assistant generation context
text += "<observation>\n" + retrieved_context + "\n</observation>\n"

print("=" * 80)
print("MODEL INPUT:")
print("=" * 80)
print(text)
print("=" * 80)

# Tokenize
inputs = tokenizer(
    text,
    return_tensors="pt",
)

# Move inputs to the device of the input embedding layer
input_device = model.get_input_embeddings().weight.device
inputs = {
    key: value.to(input_device)
    for key, value in inputs.items()
}

print(f"Input device: {input_device}")
print(f"Input tokens: {inputs['input_ids'].shape[-1]}")

# Generate
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
    )

# Extract only newly generated tokens
generated_tokens = outputs[
    0,
    inputs["input_ids"].shape[-1]:
]

# Decode
response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
)

print("=" * 80)
print("GENERATED RESPONSE:")
print("=" * 80)
print(response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


MODEL INPUT:
<|im_start|>system
You are a helpful assistant that can solve the given question step by step.
For each step, start by explaining your thought process.
If additional information is needed, the system will perform retrieval and return the top search results within <observation> and </observation>. Continue your reasoning using the retrieved information.
When you know the final answer, provide the final answer within <answer> and </answer> tags.<|im_end|>
<|im_start|>user
Question:
Where does Sarah work, and how do you know that?<|im_end|>
<|im_start|>assistant
<observation>
[1] Title: Daniel's New Job

Daniel recently started working at a small publishing company called Maple Press.
He told his friend Alex about his new job at the company.


[2] Title: Sarah and Daniel at Maple Press

Sarah works with Daniel at Maple Press.
She often helps organize book-related events for the company and works with
Daniel on book-related activities.


[3] Title: Weekend Weather Report

The 

## 3.2 Generation with Incomplete Knowledge

This experiment evaluates the generation behavior of Qwen2.5-3B-Instruct when
the model has enough information to begin reasoning but lacks the information
required to reach the final answer.

A synthetic question is used to ensure that the missing information is not
available in the model's parametric knowledge for this specific example.

No retrieved observation is provided in this experiment.

### Objectives

- Verify that the model can begin reasoning from the information available in
  the question.
- Test whether the model continues reasoning when the information required
  for the final answer is unavailable.
- Observe whether the model recognizes that additional information is needed.
- Observe whether the model stops early, produces an unsupported answer, or
  continues generating without sufficient evidence.
- Establish baseline behavior for the subsequent ART retrieval-decision
  experiments.

No ART-specific modification or retrieval is applied in this section.

In [5]:
# 3.2 Generation with Incomplete Knowledge
# Synthetic question without retrieved observation

system_prompt = """You are a helpful assistant that can solve the given question step by step.
For each step, start by explaining your thought process.
If additional information is needed, the system will perform retrieval and return the top search results within <observation> and </observation>. Continue your reasoning using the retrieved information.
When you know the final answer, provide the final answer within <answer> and </answer> tags."""

user_prompt = """Question:
Daniel works at Maple Press. His colleague Sarah is responsible for organizing
the company's historical book collection. What is the name of the collection
that Sarah is responsible for?"""

# Apply Qwen chat template up to the assistant generation point
text = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    tokenize=False,
    add_generation_prompt=True,
)

print("=" * 80)
print("MODEL INPUT:")
print("=" * 80)
print(text)
print("=" * 80)

# Tokenize
inputs = tokenizer(
    text,
    return_tensors="pt",
)

# Move inputs to the device of the input embedding layer
input_device = model.get_input_embeddings().weight.device
inputs = {
    key: value.to(input_device)
    for key, value in inputs.items()
}

print(f"Input device: {input_device}")
print(f"Input tokens: {inputs['input_ids'].shape[-1]}")

# Generate
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
    )

# Extract only newly generated tokens
generated_tokens = outputs[
    0,
    inputs["input_ids"].shape[-1]:
]

# Decode
response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
)

print("=" * 80)
print("GENERATED RESPONSE:")
print("=" * 80)
print(response)

MODEL INPUT:
<|im_start|>system
You are a helpful assistant that can solve the given question step by step.
For each step, start by explaining your thought process.
If additional information is needed, the system will perform retrieval and return the top search results within <observation> and </observation>. Continue your reasoning using the retrieved information.
When you know the final answer, provide the final answer within <answer> and </answer> tags.<|im_end|>
<|im_start|>user
Question:
Daniel works at Maple Press. His colleague Sarah is responsible for organizing
the company's historical book collection. What is the name of the collection
that Sarah is responsible for?<|im_end|>
<|im_start|>assistant

Input device: cuda:0
Input tokens: 130
GENERATED RESPONSE:
To determine the name of the collection that Sarah is responsible for, we need to consider the context provided in the question. The question states that Sarah is responsible for organizing the company's historical book col

In [6]:
# 3.3 Continue Generation After Retrieval Observation
# Synthetic question with partial reasoning followed by retrieved context

system_prompt = """You are a helpful assistant that can solve the given question step by step.
For each step, start by explaining your thought process.
If additional information is needed, the system will perform retrieval and return the top search results within <observation> and </observation>. Continue your reasoning using the retrieved information.
When you know the final answer, provide the final answer within <answer> and </answer> tags."""

user_prompt = """Question:
Daniel works at Maple Press. His colleague Sarah is responsible for organizing
the company's historical book collection. What is the name of the collection
that Sarah is responsible for?"""

partial_reasoning = """To determine the name of the collection that Sarah is
responsible for, we need to consider the information provided in the question.

The question tells us that Sarah is responsible for organizing the company's
historical book collection, but the name of the collection is not provided."""

retrieved_context = """[1] Title: Maple Press Historical Collection

Maple Press maintains a historical book collection called the Heritage
Classics Collection. Sarah is responsible for organizing this collection.


[2] Title: Maple Press Events

Sarah helps organize book-related events at Maple Press.


[3] Title: Local Weather

The city experienced heavy rainfall this week."""

# Apply Qwen chat template up to the assistant generation point
text = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    tokenize=False,
    add_generation_prompt=True,
)

# Add partial reasoning generated before retrieval
text += partial_reasoning + "\n\n"

# Inject retrieved observation
text += "<observation>\n" + retrieved_context + "\n</observation>\n"

print("=" * 80)
print("MODEL INPUT:")
print("=" * 80)
print(text)
print("=" * 80)

# Tokenize
inputs = tokenizer(
    text,
    return_tensors="pt",
)

# Move inputs to the device of the input embedding layer
input_device = model.get_input_embeddings().weight.device
inputs = {
    key: value.to(input_device)
    for key, value in inputs.items()
}

print(f"Input device: {input_device}")
print(f"Input tokens: {inputs['input_ids'].shape[-1]}")

# Generate continuation
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
    )

# Extract only newly generated tokens
generated_tokens = outputs[
    0,
    inputs["input_ids"].shape[-1]:
]

# Decode
response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
)

print("=" * 80)
print("GENERATED CONTINUATION:")
print("=" * 80)
print(response)

MODEL INPUT:
<|im_start|>system
You are a helpful assistant that can solve the given question step by step.
For each step, start by explaining your thought process.
If additional information is needed, the system will perform retrieval and return the top search results within <observation> and </observation>. Continue your reasoning using the retrieved information.
When you know the final answer, provide the final answer within <answer> and </answer> tags.<|im_end|>
<|im_start|>user
Question:
Daniel works at Maple Press. His colleague Sarah is responsible for organizing
the company's historical book collection. What is the name of the collection
that Sarah is responsible for?<|im_end|>
<|im_start|>assistant
To determine the name of the collection that Sarah is
responsible for, we need to consider the information provided in the question.

The question tells us that Sarah is responsible for organizing the company's
historical book collection, but the name of the collection is not provid